In [203]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import numpy as np

In [204]:
data = pd.read_csv('../data/simpsons_script_lines.csv')

In [205]:
data

,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,True,464.0,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,True,9.0,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,True,464.0,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,True,9.0,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5
4,9553,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,True,40.0,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33
...,...,...,...,...,...,...,...,...,...,...,...,...,...
158266,9544,32,204,Miss Hoover: (OFF LISA'S REACTION) I'm back.,831000,true,464,3.0,Miss Hoover,Springfield Elementary School,I'm back.,im back,2
158267,9545,32,205,"Miss Hoover: You see, class, my Lyme disease t...",839000,true,464,3.0,Miss Hoover,Springfield Elementary School,"You see, class, my Lyme disease turned out to ...",you see class my lyme disease turned out to be,10
158268,9546,32,206,Miss Hoover: Psy-cho-so-ma-tic.,842000,true,464,3.0,Miss Hoover,Springfield Elementary School,Psy-cho-so-ma-tic.,psy-cho-so-ma-tic,1
158269,9547,32,207,Ralph Wiggum: Does that mean you were crazy?,844000,true,119,3.0,Ralph Wiggum,Springfield Elementary School,Does that mean you were crazy?,does that mean you were crazy,6


# 1. Filtering Speaking Lines

we detect that speaking line should be a boolean, but it's not as we have lower case intances, correct the dtype 

In [206]:
data['speaking_line'] = data['speaking_line'].replace({'false': False, 'true': True}).astype(bool)

we detect that instances with speaking line = false refer to two narration, sounds of objects of the show (no character associated) or onomatopeias from the characters.
This instances won't hlp us analyzing the words of the characters and therefore we decide to delete them from the dataset to reduce it's size and keep only relevenat instances.
Since the column will be a constant after this filter, we delete it too.


In [207]:
data = data[data['speaking_line']]
data.drop(columns=['speaking_line'], inplace=True)

# 2 Incorrect Word Counts

we notice thah there are 10 incorrect rows, where the spoken words have a different format, the normalized_text is not a sequence of word, but an integer abd the word_count is true instead of an integer.
Most of this lines belong to non-main characters ot groups like 'Entire Town', singers or others.
There are a few of them that belong to the singer tonny bennet. Even though the character exists these and appears to say :Hey, good to see you these lines belong to the background song of the epsiode(https://www.youtube.com/watch?v=dZsOHz1z7Pc). Since tony bennet only says one line and the other ones are not conversation lines, we decide to rease all instances of tonny bennet.
We delte the instances from 'entire town', singers and Revue Cast Members, too, since they don't belong to individual characters and have very little represntation on the entirity of the dataset.


Moe Szyslak: "The reason I left you is simple:

In [208]:
data[data['word_count'] == 'true']

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
8082,17667,59,87,"Singers: (SINGING) ""IT'S THE FIRST ANNUAL MONT...",442000,276.0,636.0,Singers,Springfield Civic Center,IT'S THE FIRST ANNUAL MONTGOMERY BURNS/ AWARD ...,455000,true
71799,81724,282,279,"Revue Cast Members: ""WE'RE THE PERFORMERS YOU ...",1068000,3510.0,2370.0,Revue Cast Members,Branson Theater,WE'RE THE PERFORMERS YOU THOUGHT WERE DEAD / L...,1075000,true
81136,91095,315,284,"Homer Simpson: ""I-M-O--",1220000,2.0,5.0,Homer Simpson,Simpson Home,"""I-M-O--,i-m-o--,1\n91096,315,285,Homer Simpso...",1220000,true
115436,125627,445,216,"Moe Szyslak: ""The reason I left you is simple:",1095000,17.0,15.0,Moe Szyslak,Moe's Tavern,"""The reason I left you is simple:,the reason i...",1095000,true
153015,4260,14,252,"Entire Town: ""SLEIGH BELLS RING / ARE YOU LIST...",1121000,241,211.0,Entire Town,NEIGHBORHOOD,"""SLEIGH BELLS RING / ARE YOU LISTENIN'/,sleigh...",1134000,true
154078,5345,18,199,"Tony Bennett: ""THERE'S A SWINGIN' TOWN I KNOW /",993000,297,239.0,Tony Bennett,Capital City,"THERE'S A SWINGIN' TOWN I KNOW /,theres a swin...",997000,true
154080,5348,18,202,"Tony Bennett: ""PEOPLE STOP AND SCREAM HELLO /",1001000,297,239.0,Tony Bennett,Capital City,"PEOPLE STOP AND SCREAM HELLO /,people stop and...",1004000,true
154082,5351,18,205,"Tony Bennett: ""IT'S THE KIND OF PLACE THAT MAKES",1009000,297,239.0,Tony Bennett,Capital City,"IT'S THE KIND OF PLACE THAT MAKES,its the kind...",1009000,true
154084,5354,18,208,"Tony Bennett: ""AND IT MAKES A KING FEEL LIKE/",1016000,297,239.0,Tony Bennett,Capital City,"AND IT MAKES A KING FEEL LIKE/,and it makes a ...",1019000,true
156870,8152,28,13,"Gulliver Dark: (SINGING) ""THE RULES THAT CONST...",115000,187,332.0,Gulliver Dark,Aztec Theater,THE RULES THAT CONSTRAIN OTHER MEN / MEAN NOTH...,125000,true


In [209]:
data = data[~data['raw_character_text'].isin(['Tony Bennett','Singers','Revue Cast Members','Entire Town'])]

In [210]:
data[data['word_count'] == 'true']

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
81136,91095,315,284,"Homer Simpson: ""I-M-O--",1220000,2.0,5.0,Homer Simpson,Simpson Home,"""I-M-O--,i-m-o--,1\n91096,315,285,Homer Simpso...",1220000,true
115436,125627,445,216,"Moe Szyslak: ""The reason I left you is simple:",1095000,17.0,15.0,Moe Szyslak,Moe's Tavern,"""The reason I left you is simple:,the reason i...",1095000,true
156870,8152,28,13,"Gulliver Dark: (SINGING) ""THE RULES THAT CONST...",115000,187,332.0,Gulliver Dark,Aztec Theater,THE RULES THAT CONSTRAIN OTHER MEN / MEAN NOTH...,125000,true


This error comes from the transcription of the sentcne I am okay as initials I.M.O.K (Disney's subtitles also show the same)
This is actually the end of the previous sentence and not a new one. We will delete this sentence and add the correct normalized ending to the previous one and delete the incorrect one.

Also the sentence Homer says afterwards is lost. He says_  'get it? I am okay'. We will assing this sentence to the current incorrect one

In [211]:
current_sentence = 'last weeks garbage i missed the pick-up date but it doesnt matter because my mom is alive see'
missing_part = 'i am okay'
correct_sentence = current_sentence + ' ' + missing_part
word_count = len(correct_sentence.split())
data.loc[data["id"] == 91094, "word_count"] = word_count
data.loc[data["id"] == 91094, "normalized_text"] = correct_sentence

lost_sentence = 'get it i am okay'
word_count = len(lost_sentence.split())
data.loc[data["id"] == 91095, "word_count"] = word_count
data.loc[data["id"] == 91095, "normalized_text"] = lost_sentence

Moe's sentence error is also fixable, there is a missing part. 
The correct sentence is :  "The reason I left you is simple: I'm gay
We'll correct it on the data

In [212]:
correct_sentence = 'the reason i left you is simple im gay'
word_count = len(correct_sentence.split())
data.loc[data["id"] == 125627, "word_count"] = word_count
data.loc[data["id"] == 125627, "normalized_text"] = correct_sentence

The only one missing is Gulliver Dark's sentence. After checking the epsiode, we have found no appearence of the character. This means that this is most likely a background song the character is singing and not really a line in the show.
We will delete this row 

In [213]:
data = data[data['id'] != 8152]

In [214]:
data

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,464.0,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,9.0,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,464.0,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,9.0,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5
4,9553,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,40.0,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33
...,...,...,...,...,...,...,...,...,...,...,...,...
158266,9544,32,204,Miss Hoover: (OFF LISA'S REACTION) I'm back.,831000,464,3.0,Miss Hoover,Springfield Elementary School,I'm back.,im back,2
158267,9545,32,205,"Miss Hoover: You see, class, my Lyme disease t...",839000,464,3.0,Miss Hoover,Springfield Elementary School,"You see, class, my Lyme disease turned out to ...",you see class my lyme disease turned out to be,10
158268,9546,32,206,Miss Hoover: Psy-cho-so-ma-tic.,842000,464,3.0,Miss Hoover,Springfield Elementary School,Psy-cho-so-ma-tic.,psy-cho-so-ma-tic,1
158269,9547,32,207,Ralph Wiggum: Does that mean you were crazy?,844000,119,3.0,Ralph Wiggum,Springfield Elementary School,Does that mean you were crazy?,does that mean you were crazy,6


# 4 Deleting Group characters and character errors

In [215]:
sum(data['character_id'].isna()), sum(data['raw_character_text'].isna())

(2, 3)

In [216]:
data[data['raw_character_text'].isna()]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
86468,96433,334,272,: EVERYWHERE AROUND THE WORLD / THEY'RE COMIN'...,1259000,NaN,681.0,NaN,Ship,EVERYWHERE AROUND THE WORLD / THEY'RE COMIN' T...,everywhere around the world theyre comin to am...,28
142024,152461,546,20,"""ABRAHAM LINCOLN: (FURIOUS) Guess what. I also...",Springfield Elementary School,guess what i also play frankenstein,6.0,NaN,NaN,NaN,NaN,NaN
143085,153593,549,223,: Right here,1091000,NaN,503.0,NaN,Grateful Gelding Stables,Right here,right here,2


delete instances with no character

we have tow instances witch character id missmig and 3 iwth raw chacater_text misisng. The one with id and no raw is a senetnce from Abraham Lincoln (we can see in the raw text), byt the id is not created correctly. we'll do it right. We'll delete the other 2 instances

In [217]:
data.loc[[142023,142024,142025]]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
142023,152460,546,19,"""Abraham Lincoln"": Now, fellow countrymen! A h...",100000,857,3.0,Abraham Lincoln,Springfield Elementary School,"Now, fellow countrymen! A house divided agains...",now fellow countrymen a house divided against ...,8
142024,152461,546,20,"""ABRAHAM LINCOLN: (FURIOUS) Guess what. I also...",Springfield Elementary School,guess what i also play frankenstein,6.0,NaN,NaN,NaN,NaN,NaN
142025,152462,546,21,DOLPH: Douglas is gettin' away!,111000,146,3.0,DOLPH,Springfield Elementary School,Douglas is gettin' away!,douglas is gettin away,4


In [218]:
data.loc[data['id'] == 152461, 'character_id'] = 857
data.loc[data['id'] == 152461, 'raw_character_text'] = 'Abraham Lincoln'
sentence = 'guess what i also play frankenstein'
word_count = len(sentence.split())
data.loc[data['id'] == 152461, 'normalized_text'] = sentence
data.loc[data['id'] == 152461, 'word_count'] = word_count

In [219]:
data = data[data['raw_character_text'].notna()]

With the previous processing step we realized that there are instances that don't identify charaters but groups of them. There are other characters that arfe not that relevant or extras and unnamed ones.For the purpose of this assignemnt we want to focus only on the indviduall characters and therefore will delete all other instances, that augment the size of our dataset without providing any value

In [220]:
len(data[data['raw_character_text'] == "Ticket Seller"])

7

Landlady, Conductor, Husband Of Marge's Friend, Sophisticate #1, Sophisticate #2,Sophisticate #3, Sultan, Convict, Customer, Ticket Seller

potser no fa falta is implement ens quedem arbitrariament alguns o top 10-15 amb més frases

In [221]:
#sorted(data['raw_character_text'].unique().tolist())


# Correcting word_count columns

There are 4 sentences where the spoken_words, normalized_text and word_count columns don't make sense, we'll delete this instances to have a consistent format and not lose any meaningfull data impotance.

In [222]:
data

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,464.0,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,9.0,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,464.0,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,9.0,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5
4,9553,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,40.0,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33
...,...,...,...,...,...,...,...,...,...,...,...,...
158266,9544,32,204,Miss Hoover: (OFF LISA'S REACTION) I'm back.,831000,464,3.0,Miss Hoover,Springfield Elementary School,I'm back.,im back,2
158267,9545,32,205,"Miss Hoover: You see, class, my Lyme disease t...",839000,464,3.0,Miss Hoover,Springfield Elementary School,"You see, class, my Lyme disease turned out to ...",you see class my lyme disease turned out to be,10
158268,9546,32,206,Miss Hoover: Psy-cho-so-ma-tic.,842000,464,3.0,Miss Hoover,Springfield Elementary School,Psy-cho-so-ma-tic.,psy-cho-so-ma-tic,1
158269,9547,32,207,Ralph Wiggum: Does that mean you were crazy?,844000,119,3.0,Ralph Wiggum,Springfield Elementary School,Does that mean you were crazy?,does that mean you were crazy,6


In [223]:
wrong_indexes = [152968,130608,117571,86744]
data.loc[wrong_indexes][['raw_character_text','spoken_words', 'normalized_text', 'word_count']]


,raw_character_text,spoken_words,normalized_text,word_count
152968,Thomas Jefferson,"""We hold these truths to be self-evident.,we h...",the humiliation,the fact that it wasn't me. I've never felt so...
130608,Homer Simpson,"""Override self-destruct protocol with authoriz...",THEN:) Uh,"just don't ask me to drive you to the airport."""
117571,Marge Simpson,"""The patrollers were too fast for Eliza and Vi...",a little schmear,and presto -- you're part of the under-clown r...
86744,Patty Bouvier,"""Are you a 'Patty' or a 'Selma?' Take our quiz...",blow me down,"I'm a Selma."""


In [224]:
data = data.drop(index=wrong_indexes)


# Better naming and labeling

??mapejar 10 yr old hommer -> hommer??

# 5 Simplifying labels

we can relabel the characters so they occupy less space in future visualizations, while still being able to identify them.
ex Homer Simposn -> Homer
A version of the same character is categorized using different value for column `raw_character_text`. In some cases the identifier `character_id` would  help us mapping them together, but not it does not group all versions into a single id.
We can find an example of this below


In [225]:
characters_data = data.drop_duplicates(subset='raw_character_text')[['character_id','raw_character_text']]
characters_data[characters_data['character_id'].isin([2,918,2100])]


,character_id,raw_character_text
57,2.0,Homer Simpson
10134,918.0,Homer
13593,2.0,Homer's Brain
35375,2100.0,Flashback Homer
79155,2.0,Homer's Thoughts
106766,2.0,Young Homer


In [226]:
characters_data['character_id'].value_counts()

character_id
2.0       4
9.0       4
1.0       4
8.0       3
603.0     2
         ..
2480.0    1
2479.0    1
2478.0    1
2477.0    1
468       1
Name: count, Length: 6248, dtype: int64

That is why we will map these different versions into a single character 
ex '1-Year-Old Bart' -> Bart
yokel boy yokel girl qui son? Els gitanaos?

s16 epsidoe 2 -> 2nd hommer, don't know what to do

# Dealin with missing values on the normalized text

there are a few instances that belong to noises, and they don't produce any words. We'll delete them from the dataset

In [227]:
data = data[data['normalized_text'].notna()]

# Dealing with strings on the normalized text

some rows have -- on the normalized words column. This belong to sentences getting interrupted, but the word count column counts it as a word.
ex now kirk its only a game sometimes we --  : https://www.youtube.com/watch?v=DXeRCJzrQEA

We wi'll delete the '-' from the normalized_text and recount the words for them

In [228]:
indexes = data[data['normalized_text'].str.contains('-')].index
data.loc[indexes, 'normalized_text'] = data.loc[indexes, 'normalized_text'].str.replace('-', ' ', regex=False)
data.loc[indexes, 'word_count'] = data.loc[indexes, 'normalized_text'].str.split().apply(len)

In [229]:
data[data['normalized_text'].str.contains('--')]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count


# Dealing with outliers

We find and instance with a suposed word count of 571000. The issue we see is that the spoken words are part of the lyrics of the 'Wannabe' song, while the normalzied text references a sentence that Bart said. The Wanabee song sounds in the begining of the scene of the sentence so it both got mixed up in a single instance. We'll correct it to reflect Bart's sentence. We alos normalize the text since it's not

In [230]:
data.loc[52605]

id                                                                62483
episode_id                                                          220
number                                                              100
raw_text              ABBA: "If you wanna be my lover / You gotta ge...
timestamp_in_ms                                                  551000
character_id                                                     2739.0
location_id                                                      1902.0
raw_character_text                                                 ABBA
raw_location_text                                           Red's Truck
spoken_words          "If you wanna be my lover / You gotta get with...
normalized_text                   Dad. He wants you to blow your horn."
word_count                                                       571000
Name: 52605, dtype: object

In [231]:
data.loc[data['id'] == 62483,'raw_text'] = None
data.loc[data['id'] == 62483,'spoken_words'] = None
data.loc[data['id'] == 62483,'raw_character_text'] = 'Bart'
sentence = 'dad he wants you to blow your horn.'    
data.loc[data['id'] == 62483,'normalized_text'] = sentence
word_count = len(sentence.split())
data.loc[data['id'] == 62483,'word_count'] = word_count

"Impossible to tell in writing. 'Bashõ',impossible to tell in writing bashõ,6
83466,289,165,"Robert Pinsky: He named himself

Robert Pinsky: "Impossible to tell in writing. 'Bashõ'

Robert pinsky also has a sentence with 672000 word_count. All the sentences by him belong to a single epsiode where he reads a poem. Only parts ot the poem appear, and the one with the mistaken word counts has one the last sentence of the first paragaph and the first one of the following one in columns spoken_words and normalized_text respectively, when the whole correct sentence is 'impossible to tell in writing bashõ He named himself 'Banana Tree'.
We'll correct it

In [232]:
data.loc[73537]

id                                                                83465
episode_id                                                          289
number                                                              164
raw_text              Robert Pinsky: "Impossible to tell in writing....
timestamp_in_ms                                                  670000
character_id                                                     3592.0
location_id                                                      2425.0
raw_character_text                                        Robert Pinsky
raw_location_text                                            Café Kafka
spoken_words          "Impossible to tell in writing. 'Bashõ',imposs...
normalized_text                                     'Banana Tree'..."""
word_count                                                       672000
Name: 73537, dtype: object

In [233]:
sentence = "impossible to tell in writing bashõ He named himself banana tree"
word_count = len(sentence.split())
data.loc[73537,'normalized_text'] = sentence
data.loc[73537,'word_count'] = word_count

Bart also has an outlier 

In [234]:
data[data['id'] == 5432]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
154163,5432,19,5,"Bart Simpson: (WRITING) ""One o'clock -- still ...",104000,8,5.0,Bart Simpson,Simpson Home,"""One o'clock -- still just a potato.,one ocloc...",neighbor! The Lord's certainly given us a beau...,108000


In [235]:
sentence = "one o clock still just a potato neighbor the lord is certainly given us a beautiful day today huh?"
wprd_count = len(sentence.split())
data.loc[data['id'] == 5432, 'normalized_text'] = sentence
data.loc[data['id'] == 5432, 'word_count'] = wprd_count

marge outlier

In [236]:
data[data['id'] == 69807]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
59908,69807,243,292,"Marge Simpson: (CONTINUING) ""My gold is in the...",1145000,1.0,422.0,Marge Simpson,White House,"""My gold is in the heart of every freedom-lovi...","crap!""",1145000


In [237]:
sentence = 'my gold is in the heart of every freedom loving american'
word_count = len(sentence.split())
data.loc[data['id'] == 69807, 'normalized_text'] = sentence
data.loc[data['id'] == 69807, 'word_count'] = word_count

Homer otulier

In [238]:
data.loc[data['id'] == 88907]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
78951,88907,308,85,"Homer Simpson: (READING) ""Nausea... cravings.....",393000,2.0,15.0,Homer Simpson,Moe's Tavern,"""Nausea... cravings... knocked-up feeling... S...",the popular singer slash songwriter slash puzz...,409000


the popular singer slash songwriter slash puzzle piece."

In [239]:
sentence = "nausea cravings knocked up feeling she was pregnant with bart and that is the reason she stayed with me"
word_count = len(sentence.split())
data.loc[data['id'] == 88907, 'normalized_text'] = sentence
data.loc[data['id'] == 88907, 'word_count'] = word_count

Lisa outlier 1

In [240]:
data.loc[data['id'] == 87170]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
77228,87170,302,13,"Lisa Simpson: (READING) ""Molochai desiratum ma...",147000,9.0,5.0,Lisa Simpson,Simpson Home,"""Molochai desiratum maledictu... nosferatu asc...","Mad libs!""",147000


In [241]:
sentence = 'molochai desiratum maledictu nosferatu ascendum corporalis'
word_count = len(sentence.split())
data.loc[data['id'] == 87170, 'normalized_text'] = sentence
data.loc[data['id'] == 87170, 'word_count'] = word_count

lisa outlier 2

In [242]:
data.loc[data['id'] == 111237]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
101152,111237,390,16,"Lisa Simpson: ""...Hitachee Tribe. (NERVOUS GIG...",117000,9.0,6.0,Lisa Simpson,KITCHEN,"""...Hitachee Tribe.,hitachee tribe,2\n111238,3...",is it wrong for me to appropriate the culture ...,14


In [243]:
sentence = 'is it wrong for me to appropriate the culture of a long suffering people'
word_count = len(sentence.split())
data.loc[data['id'] == 111237, 'normalized_text'] = sentence
data.loc[data['id'] == 111237, 'word_count'] = word_count

# 3 Irrelevant Columns

there are a few columns that are also irrelevant for our use case.
* timestamp_in_ms -> we don't care about the moment in the episode the character speaks
* location_id -> we don't care about location
* raw_location_text -> same as the one above
* raw_text -> same as normalized text and spoken words, but adds noise such as emotions between brackets, which we are not interested in this assingment and make processing more complicated
* spoken_words -> it's repetitive having it's normalized version 

We can delete this columns


In [244]:
data.drop(columns = ['timestamp_in_ms','location_id','raw_location_text','raw_text','spoken_words'],inplace=True)

# 4 Adding Season and number in season

we don't have a clear column indicating the season the epsiodes belong to, even though they can be deuced from the epsiode_id column, which indicates the epsiode without distingushing between seasons.
Since maaping this epsidoes to their respective seasons would requiere manually checking each one of them, we can use the dataset of the previous assingment that already has a defined 'season' column to create the column in our dataset

Since for the clean version of the dataset of the previous assignment we deleted all epsiodes of the last available season (it only had 4 episdes) to help our interests, we won't use this version for this assingment since we can use that information. We will read the raw one

In [245]:
data_seasons = pd.read_csv('../data/simpsons_episodes_raw.csv')[['season','number_in_season','number_in_series']]

In [246]:
episodes_daat1 = set(data['episode_id'])
episodes_data2 = set(data_seasons['number_in_series'])
episodes_daat1 - episodes_data2

set()

we see that all epsiodes in our script lines dataset are in the datset from the previous quarter

In [247]:
print(episodes_data2 - episodes_daat1)
len(episodes_data2 - episodes_daat1)

{550, 424, 569, 441, 570, 571, 572, 573, 575, 576, 577, 578, 579, 580, 581, 582, 583, 574, 447, 584, 585, 586, 587, 588, 589, 590, 591, 592, 593, 594, 595, 596, 597, 598, 599, 600}


36

However this is not true when inverting the order, the dataset from the previous assignment has 32 more episodes. The second source used on the previous assignment also has this same issue. We explored extra datasets such as this one: https://github.com/jcrodriguez1989/thesimpsons or https://github.com/rfordatascience/tidytuesday/blob/main/data/2025/2025-02-04/readme.md but to no success. No datasets contained the missing epsiodes

In [248]:
data = data.merge(data_seasons, left_on='episode_id', right_on='number_in_series', how='left')
data.drop(columns=['number_in_series'], inplace=True)
cols = data.columns.tolist()
cols.insert(cols.index('episode_id'), cols.pop(cols.index('season')))
cols.insert(cols.index('episode_id'), cols.pop(cols.index('number_in_season')))
data = data[cols]
data

,id,season,number_in_season,episode_id,number,character_id,raw_character_text,normalized_text,word_count
0,9549,2,19,32,209,464.0,Miss Hoover,no actually it was a little of both sometimes ...,31
1,9550,2,19,32,210,9.0,Lisa Simpson,wheres mr bergstrom,3
2,9551,2,19,32,211,464.0,Miss Hoover,i dont know although id sure like to talk to h...,22
3,9552,2,19,32,212,9.0,Lisa Simpson,that life is worth living,5
4,9553,2,19,32,213,40.0,Edna Krabappel-Flanders,the polls will be open from now until the end ...,33
...,...,...,...,...,...,...,...,...,...
132016,9544,2,19,32,204,464,Miss Hoover,im back,2
132017,9545,2,19,32,205,464,Miss Hoover,you see class my lyme disease turned out to be,10
132018,9546,2,19,32,206,464,Miss Hoover,psy cho so ma tic,5
132019,9547,2,19,32,207,119,Ralph Wiggum,does that mean you were crazy,6


# 5 Renamiming to keep columns undestandable

column number identifies the line within each, season. We find this name rather confusing, we'll change it to one that explains the variable better

In [249]:
data.rename(columns={'number': 'line_number_in_episode'}, inplace=True)  

In [250]:
data.to_csv('../data/simpsons_script_lines_cleaned.csv', index=False)

In [251]:
data

,id,season,number_in_season,episode_id,line_number_in_episode,character_id,raw_character_text,normalized_text,word_count
0,9549,2,19,32,209,464.0,Miss Hoover,no actually it was a little of both sometimes ...,31
1,9550,2,19,32,210,9.0,Lisa Simpson,wheres mr bergstrom,3
2,9551,2,19,32,211,464.0,Miss Hoover,i dont know although id sure like to talk to h...,22
3,9552,2,19,32,212,9.0,Lisa Simpson,that life is worth living,5
4,9553,2,19,32,213,40.0,Edna Krabappel-Flanders,the polls will be open from now until the end ...,33
...,...,...,...,...,...,...,...,...,...
132016,9544,2,19,32,204,464,Miss Hoover,im back,2
132017,9545,2,19,32,205,464,Miss Hoover,you see class my lyme disease turned out to be,10
132018,9546,2,19,32,206,464,Miss Hoover,psy cho so ma tic,5
132019,9547,2,19,32,207,119,Ralph Wiggum,does that mean you were crazy,6


In [252]:
images = {
    'Marge Simpson': 'https://img.icons8.com/doodle/48/marge-simpson.png',
    'Homer Simpson': 'https://img.icons8.com/doodle/48/homer-simpson.png',
    'Bart Simpson': 'https://img.icons8.com/doodle/48/bart-simpson.png',
    'Lisa Simpson': 'https://img.icons8.com/doodle/48/lisa-simpson.png',
    'C. Montgomery Burns': 'https://img.icons8.com/doodle/48/charles-montgomery-burns.png',
    'Moe Szyslak': 'https://img.icons8.com/doodle/48/bartender-mo.png',
    'Frank Grimes': 'https://static.simpsonswiki.com/images/a/a0/Tapped_Out_Frank_Grimes_Icon_-_Relaxed.png',
    'Coyote': 'https://static.simpsonswiki.com/images/7/73/Tapped_Out_Space_Coyote_Icon_-_Happy.png',
    'Milhouse Van Houten': 'https://img.icons8.com/doodle/48/milhouse-van-houten.png',
    'Krusty the Clown':'https://img.icons8.com/doodle/48/krusty-the-clown.png',
    'Ralph Wiggum': 'https://img.icons8.com/doodle/48/ralf.png',
    'Chief Wiggum':'https://img.icons8.com/doodle/48/clancy-wiggum.png',
    'Ned Flanders':'https://img.icons8.com/doodle/48/ned-flanders.png',
    'Barney Gumble': 'https://img.icons8.com/doodle/48/barney-gumble.png',
    'Grampa Simpson': 'https://img.icons8.com/doodle/48/abraham-simpson.png',
    'Apu Nahasapeemapetilon': 'https://static.simpsonswiki.com/images/0/08/Tapped_Out_Apu_Icon_-_Happy.png',
    'Nelson Muntz':'https://static.simpsonswiki.com/images/0/0c/Tapped_Out_Nelson_Icon_-_Brave.png',
    'Groundskeeper Willie':'https://static.simpsonswiki.com/images/4/40/Tapped_Out_Chainsaw_Willie_Icon_-_Happy.png',
    'Duffman':'https://static.simpsonswiki.com/images/1/18/Tapped_Out_Duffman_Icon.png',
    'Dr. Julius Hibbert':'https://static.simpsonswiki.com/images/d/d5/Tapped_Out_Dr._Hibbert_Icon.png',
    'Edna Krabappel-Flanders':'https://static.simpsonswiki.com/images/1/14/Tapped_Out_Mrs._Krabappel_Icon.png',
    'Sideshow Bob':'https://static.simpsonswiki.com/images/6/68/Tapped_Out_Sideshow_Bob_Icon_-_Serious.png',
    'Waylon Smithers':'https://static.simpsonswiki.com/images/3/3a/Tapped_Out_Smithers_Icon.png',
    'Lenny Leonard':'https://static.simpsonswiki.com/images/6/6f/Tapped_Out_Lenny_Icon.png',
    'Carl Carlson':'https://static.simpsonswiki.com/images/a/a4/Tapped_Out_Carl_Icon.png',
    'Kent Brockman':'https://static.simpsonswiki.com/images/3/35/Tapped_Out_Brockman_Sidebar.png',
    'Rev. Timothy Lovejoy':'https://static.simpsonswiki.com/images/4/40/Tapped_Out_Rev._Lovejoy_Sidebar.png',
    'Mayor Joe Quimby':'https://static.simpsonswiki.com/images/e/e3/Tapped_Out_Quimby_Icon.png',
    'Comic Book Guy':'https://static.simpsonswiki.com/images/a/a9/Tapped_Out_Comic_Book_Guy_Icon.png',
    'Gary Chalmers':'https://static.simpsonswiki.com/images/5/58/Tapped_Out_Chalmers_Icon.png',
    'Selma Bouvier':'https://static.simpsonswiki.com/images/8/8f/Tapped_Out_Selma_Icon.png',
    'Professor Jonathan Frink':'https://static.simpsonswiki.com/images/0/09/Tapped_Out_Professor_Frink_Icon_-_Deadpan.png',
    'Patty Bouvier':'https://static.simpsonswiki.com/images/3/31/Tapped_Out_Patty_Sidebar.png',
    'Martin Prince':'https://static.simpsonswiki.com/images/d/da/Tapped_Out_Martin_Icon.png',
    'Otto Mann':'https://static.simpsonswiki.com/images/4/48/Tapped_Out_Conductor_Otto_Sidebar.png',
    'Sideshow Mel':'https://static.simpsonswiki.com/images/6/6d/Tapped_Out_Sideshow_Mel_Icon.png',
    'Fat Tony':'https://static.simpsonswiki.com/images/1/1b/Tapped_Out_Fat_Tony_Icon.png',
    'Lou':'https://static.simpsonswiki.com/images/c/cf/Tapped_Out_Lou_Icon.png',
    'Jimbo Jones':'https://static.simpsonswiki.com/images/f/f3/Tapped_Out_Jimbo_Icon.png',
    'Kirk Van Houten':'https://static.simpsonswiki.com/images/6/61/Tapped_Out_Acorn_Kirk_Icon.png',
    'Agnes Skinner':'https://static.simpsonswiki.com/images/b/b1/Tapped_Out_Agnes_Sidebar.png',
    'Cletus Spuckler': 'https://static.simpsonswiki.com/images/a/a0/Tapped_Out_Cletus_Icon_-_Smiling.png'
}

In [253]:
data['word_count'] = pd.to_numeric(data['word_count'], errors='coerce').astype(float)

In [254]:
data_line_plot = data[data['raw_character_text'].isin(images.keys())]
data_line_plot = data_line_plot.groupby(['season','raw_character_text'])['word_count'].sum().reset_index()
data_line_plot = data_line_plot[data_line_plot['raw_character_text'].isin(images.keys())]
data_line_plot.to_csv('../data/simpsons_script_lines_line_plot.csv', index=False)